## Trading signal construction from limit order book data





The following code downloads the limit order book data used in the lecture.

In [ ]:
# the course downloads data_full.zip (the link is in the course notebooks) and unzips it.
# I keep it in data/ (gitignored, the files are course material); data/ or data/data_full/ both work
from pathlib import Path

folders = [Path("..") / "data", Path("..") / "data" / "data_full"]
for name in ["Data_A.csv", "Data_B.csv"]:
    found = any((folder / name).exists() for folder in folders)
    print(f"{name}: {'found' if found else 'MISSING, unzip data_full.zip into data/'}")

# Loading and examinig the limit order book data

The data for training and testing is contained in two CSV files _Data_A_ and _Data_B_.

_Data_A_ contains a $100\,000 \times 22$ array with the following information:

- Column 1: the label --- midprice change direction (we define midprice $=\frac{\text{bid price}+\text{ask price}}{2}$) coded as follows: $0$ means down, $1$ means up.
- Columns 2--22: the features, all recorded just prior to the midprice change corresponding to the label.

    - Column 2: Sell side, limit order book level 1, Price (in US dollars multiplied by $10,000$), that is, the _ask price_.
    - Column 3: Sell side, limit order book level 1, Volume (in number of shares).
    - Column 4: Buy side, limit order book level 1, Price, that is, the _bid price_.
    - Column 5: Buy side, limit order book level 1, Volume.
    - Column 6: Sell side, limit order book level 2, Price.
    - Column 7: Sell side, limit order book level 2, Volume.
    - Column 8: Buy side, limit order book level 2, Price.
    - Column 9: Buy side, limit order book level 2, Volume.
    - Column 10: Sell side, limit order book level 3, Price.
    - Column 11: Sell side, limit order book level 3, Volume.
    - Column 12: Buy side, limit order book level 3, Price.
    - Column 13: Buy side, limit order book level 3, Volume.
    - Column 14: Sell side, limit order book level 4, Price.
    - Column 15: Sell side, limit order book level 4, Volume.
    - Column 16: Buy side, limit order book level 4, Price.
    - Column 17: Buy side, limit order book level 4, Volume.
    - Columns 18--22: five previous midprice change directions _0_/_1_-coded like the labels).


The rows of this file have been _randomly drawn_ from a larger data set covering the period 1 August -- 30 October 2023, and they can be treated as $100\,000$ independent samples. No _time series structure_ can be recovered from the data.


$\textit{Data_B}$ contains a $10\,000 \times 22$ array with data drawn in the same way. We will use it to assess the out of sample performance.

In this notebook we will do:
- Build and train a _binary classifier_ that predicts the label in the first column of _Data_A_.
- Use the binary classifier to predict the labels missing from _Data_B_.

We start by importing relevant packages.

In [ ]:
import sys
sys.path.append('..')  # so we can import from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

from src.data import load_lob
from src.features import add_lob_features

Next we read the data and print information to ensure it has been properly loaded.

In [ ]:
# load_lob gives the 22 columns readable names: label, ask1_price, ask1_vol, bid1_price, ...
data_A = load_lob("Data_A")   # training + validation
data_B = load_lob("Data_B")   # kept aside for the final out of sample test
print(data_A.shape, data_B.shape)

In [ ]:
data_A.info()

In [ ]:
data_A.head()

In [ ]:
# a few sanity checks before modelling
# 1. are the classes balanced? if 60% of labels were "up", 60% accuracy would be free
print("share of up moves in A:", round(data_A["label"].mean(), 4))
print("share of up moves in B:", round(data_B["label"].mean(), 4))

# 2. the book should never be crossed: best ask must be above best bid
print("crossed books:", (data_A["ask1_price"] <= data_A["bid1_price"]).sum())

# 3. orders of magnitude of each kind of column
data_A.describe().loc[["mean", "std", "min", "max"]].T.iloc[[1, 2, 3, 4, 17]]

Split the data into features and labels.

We also set aside part of the data for validating. These data points are not used for training, but instead they allow us to assess how well the trained network performs out-of-sample.

In [ ]:
# features = everything except the label (first column)
X = data_A.drop(columns="label").values
y = data_A["label"].values

# first 80,000 rows to train, last 20,000 to validate (same as the lecture)
# the rows are independent random draws, so this is as good as a random split
SPLIT = 80_000
X_train, X_val = X[:SPLIT], X[SPLIT:]
y_train, y_val = y[:SPLIT], y[SPLIT:]

X_test = data_B.drop(columns="label").values
y_test = data_B["label"].values
print(X_train.shape, X_val.shape, X_test.shape)

#### Question: Have a look at the data above. Do you see anything that we should be careful with?

----

**My answer:** three things stand out.

1. **Scale.** Prices are around $10^6$ (dollars x 10,000), volumes are in the hundreds and the previous directions are 0 or 1. Without scaling, gradient based fitting and any penalty on the coefficients are dominated by the price columns.
2. **The prices are absolute levels.** They drift over the three months, so the raw ask price says little about the next move. What should matter is the *shape* of the book: the spread, and how the volume is split between bid and ask. I come back to this in the challenge at the end.
3. **The five previous directions are binary**, so squaring them later does nothing on its own ($0^2 = 0$, $1^2 = 1$); only products with other features can add information.

Different features have different orders of magnitude (prices are of order $100'000$, volumes of order $100$). To make sure this does not create any issues we normalize the data.

In [ ]:
# fit the scaler on the TRAINING data only, then apply the same transform everywhere
# (fitting on validation or test data would leak information about them)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

In [ ]:
# check: training columns should now have mean 0 and std 1
print("means:", X_train_s.mean(axis=0).round(2)[:6])
print("stds: ", X_train_s.std(axis=0).round(2)[:6])

# Training a logistic regression model

We now train a logistic regression model.

In [ ]:
logit = LogisticRegression(max_iter=1000)
logit.fit(X_train_s, y_train)

print(f"train accuracy:      {accuracy_score(y_train, logit.predict(X_train_s)):.4f}")
print(f"validation accuracy: {accuracy_score(y_val, logit.predict(X_val_s)):.4f}")

# which features does it lean on? (coefficients are comparable because the inputs are scaled)
coefs = pd.Series(logit.coef_[0], index=data_A.columns[1:])
coefs.sort_values().plot.barh(figsize=(6, 6), title="logistic regression coefficients")
plt.show()

A simple logistic regression model yields 72\% on both training and validation sets.

One could also  try to add non-linear functions of the features into the regression in order to get higher accuracy. For example, one could include the squares of all features.

In [ ]:
# all degree 2 terms: every feature, its square, and every product of two features
# 21 features -> 21 + 21 squares + 210 pairs = 252 inputs (same as the lecture)
logit_sq = make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         LogisticRegression(solver="newton-cg", max_iter=1000))
logit_sq.fit(X_train_s, y_train)

print(f"train accuracy:      {logit_sq.score(X_train_s, y_train):.4f}")
print(f"validation accuracy: {logit_sq.score(X_val_s, y_val):.4f}")

By including quadratic features, the logistic regression achieves better accuracy, particuarly  in the validation set. Hence we believe the quadratic model will outperform the linear model, even in an out-of-sample scenario.

Finally we now assess the out-of-sample performance of your network.

In [ ]:
# out of sample: Data_B has never been touched, we only look at it once at the very end
acc_lin = accuracy_score(y_test, logit.predict(X_test_s))
acc_sq = logit_sq.score(X_test_s, y_test)
print(f"Test accuracy of linear model:    {acc_lin:.4f}")
print(f"Test accuracy of quadratic model: {acc_sq:.4f}")

In [ ]:
# accuracy alone can hide a model that just predicts "up" a lot, so look at the confusion matrix
cm = confusion_matrix(y_test, logit_sq.predict(X_test_s))
pd.DataFrame(cm, index=["true down", "true up"], columns=["pred down", "pred up"])

#### Challenge: can you further improve the validation accuracy?

Possible points you may try:

- Regularization
- High-order features
- Other methods, say decision trees?


In [ ]:
# Challenge. I try four ideas, comparing them on the VALIDATION set only
# (choosing on the test set would make the test score optimistic)

# 1. is it the squares or the products of pairs that help?
square = lambda A: np.hstack([A, A**2])   # squares only: 21 -> 42 inputs
m = LogisticRegression(max_iter=1000).fit(square(X_train_s), y_train)
print(f"squares only, validation accuracy {accuracy_score(y_val, m.predict(square(X_val_s))):.4f}")

# 2. regularisation strength on the full degree 2 model: smaller C = stronger penalty
for C_ in [0.01, 0.1, 1, 10]:
    m = make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                      LogisticRegression(C=C_, solver="newton-cg", max_iter=1000)).fit(X_train_s, y_train)
    print(f"degree 2, C={C_:<5} validation accuracy {m.score(X_val_s, y_val):.4f}")

In [ ]:
# 3. features a trader would actually look at: spread, order book imbalance and level gaps
#    (see add_lob_features in src/features.py for the definitions)
feat_A = add_lob_features(data_A)
feat_B = add_lob_features(data_B)
new_cols = ["spread", "imbalance_1", "imbalance_all", "ask_gap", "bid_gap"]

# quick look: how does the chance of an up move change with level 1 imbalance?
bins = pd.cut(feat_A["imbalance_1"], 10)
feat_A.groupby(bins, observed=True)["label"].mean().plot(marker="o")
plt.ylabel("share of up moves")
plt.xlabel("level 1 imbalance (bid heavy on the right)")
plt.show()

In [ ]:
# same split as before (the same rows)
XE = feat_A.drop(columns="label").values
XE_train, XE_val = XE[:SPLIT], XE[SPLIT:]
XE_test = feat_B.drop(columns="label").values

scaler_E = StandardScaler().fit(XE_train)
XE_train_s, XE_val_s, XE_test_s = (scaler_E.transform(A) for A in (XE_train, XE_val, XE_test))

logit_E = make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                        LogisticRegression(solver="newton-cg", max_iter=1000)).fit(XE_train_s, y_train)
print(f"degree 2 + book features, validation accuracy {logit_E.score(XE_val_s, y_val):.4f}")

In [ ]:
# 4. a tree based model, which finds non linear effects and interactions by itself
from sklearn.ensemble import HistGradientBoostingClassifier

gbm = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0)
gbm.fit(XE_train, y_train)   # trees do not need scaled inputs
print(f"gradient boosted trees, validation accuracy {accuracy_score(y_val, gbm.predict(XE_val)):.4f}")

In [ ]:
# finally, the out of sample score for each challenger, next to the two original models
summary = pd.Series({
    "logistic, linear": acc_lin,
    "logistic, degree 2": acc_sq,
    "logistic, degree 2 + book features": logit_E.score(XE_test_s, y_test),
    "gradient boosted trees + book features": accuracy_score(y_test, gbm.predict(XE_test)),
})
summary.round(4).to_frame("test accuracy (Data_B)")